# Preloaded Map-Style Dataset — Solution

In [10]:
import torch
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
N, D = 1000, 10
X_data = torch.randn(N, D)
y_data = X_data @ torch.randn(D) + 0.1 * torch.randn(N)

In [ ]:
class PreloadedDataset(Dataset):
    """Map-style dataset that holds all data in memory as tensors."""

    def __init__(self, X: torch.Tensor, y: torch.Tensor):
        # as_tensor accepts tensors, numpy arrays and lists, and avoids a copy when it can
        self.X = torch.as_tensor(X)
        self.y = torch.as_tensor(y)
        assert len(self.X) == len(self.y), "X and y must have the same number of samples"

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


ds = PreloadedDataset(X_data, y_data)
loader = DataLoader(ds, batch_size=16, shuffle=True)

for batch_x, batch_y in loader:
    print(f"Batch: X={batch_x.shape}, y={batch_y.shape}")
    break

In [ ]:
# === Tests ===

# Test 1: Basic dataset properties
ds = PreloadedDataset(X_data, y_data)
assert len(ds) == N, f"Expected length {N}, got {len(ds)}"
x_i, y_i = ds[0]
assert x_i.shape == (D,), f"Expected feature shape ({D},), got {x_i.shape}"
assert y_i.shape == (), f"Expected scalar target, got shape {y_i.shape}"
x_7, y_7 = ds[7]
assert torch.equal(x_7, X_data[7]) and torch.equal(y_7, y_data[7]), "ds[i] must return (X[i], y[i])"
ds_np = PreloadedDataset(X_data.numpy(), y_data.numpy())  # numpy input is stored as tensors too
assert all(isinstance(t, torch.Tensor) for t in ds_np[0]), "Items should be tensors even when X, y are numpy arrays"
print("PASSED: Dataset has correct length, item shapes and (x_i, y_i) pairing")

# Test 2: DataLoader batching
loader = DataLoader(ds, batch_size=16, shuffle=False)
batch_x, batch_y = next(iter(loader))
assert batch_x.shape == (16, D), f"Batch X shape wrong: {batch_x.shape}"
assert batch_y.shape == (16,), f"Batch y shape wrong: {batch_y.shape}"
print("PASSED: DataLoader produces correct batch shapes")

# Test 3: Full epoch — every sample seen exactly once
loader_shuffled = DataLoader(ds, batch_size=16, shuffle=True)
all_y = []
for bx, by in loader_shuffled:
    all_y.append(by)
all_y = torch.cat(all_y)
assert len(all_y) == N, f"Expected {N} samples in one epoch, got {len(all_y)}"
assert torch.allclose(all_y.sort().values, y_data.sort().values), \
    "Not all samples were seen — values don't match after sorting"
print("PASSED: Full epoch sees every sample exactly once")

# Test 4: Shuffled order differs from original
loader_ordered = DataLoader(ds, batch_size=N, shuffle=False)
y_ordered = next(iter(loader_ordered))[1]
assert not torch.equal(all_y, y_ordered), "Shuffled loader should produce a different order"
print("PASSED: Shuffled order differs from sequential order")

print("\nAll tests passed!")

**Going further: batched indexing.** By default `DataLoader` calls `__getitem__` once per sample and then stacks the results in `collate_fn`. When the data is already one big in-memory tensor, it is faster to index a whole batch at once: pass a `BatchSampler` as the sampler and set `batch_size=None` (which turns automatic batching off), so `__getitem__` receives a list of indices and `self.X[idx]` returns the batch directly:

```python
from torch.utils.data import BatchSampler, RandomSampler
loader = DataLoader(ds, sampler=BatchSampler(RandomSampler(ds), batch_size=16, drop_last=False), batch_size=None)
```

`torch.utils.data.TensorDataset` is the built-in version of this class.